# Daily Practice — 2026-11-03 — ML Testing & MLOps: Data Validation, Metric Blind Spots, and Drift Detection

**Dataset:** [Breast Cancer Wisconsin (Diagnostic)](https://scikit-learn.org/stable/datasets/toy_dataset.html#breast-cancer-wisconsin-diagnostic-dataset) —
569 real patient tumor samples with 30 numeric features derived from digitized cell-nuclei images,
loaded via `sklearn.datasets.load_breast_cancer`. This is the classic real-world UCI dataset, not a
synthetic one.

## Problem statement

You've inherited a binary classifier (malignant vs. benign) built on this dataset. Your job is **not**
to improve its accuracy — it's to build the automated test suite a quality engineer would wire into a
CI/release pipeline to catch it breaking *in production*, long after a data scientist stopped looking
at it. Three failure modes you need to guard against:

1. **Bad input data** silently reaching the model (wrong dtypes, out-of-range values, missing fields).
2. **A misleading headline metric.** A classifier can report 95%+ accuracy while quietly missing most
   of the positive (malignant) cases on a skewed batch of traffic — the "accuracy paradox" that a
   single aggregate number hides.
3. **Silent feature drift.** The distribution of incoming feature values can drift away from what the
   model was trained on (sensor recalibration, a new lab's measurement conventions, a population shift)
   long before anyone retrains or even notices.

To exercise failure modes (2) and (3) under controlled conditions, you'll test against two *synthetic*
fixtures built **on top of** the real data: an artificially class-imbalanced batch (to expose the
accuracy paradox) and an artificially drifted batch (features shifted, to expose drift). Injecting these
synthetically is necessary here — you can't wait for real production drift to show up to test whether
your drift detector works — but the underlying data and model are real.

## What you should produce

1. A `DataValidator` with three methods — `validate_schema(df)`, `validate_ranges(df, feature_ranges)`,
   `validate_missing(df)` — each returning a **list of violation strings** (empty list = pass).
2. An `evaluate_model(model, X, y)` function returning a dict with overall `accuracy`, per-class
   `precision`/`recall`/`f1` (use `classification_report(..., output_dict=True)`), and the confusion
   matrix — then a demonstration, on the artificially imbalanced fixture, of how overall accuracy can
   look fine (>90%) while malignant-class recall collapses.
3. A `detect_drift(reference_df, current_df, alpha=0.05)` function that runs a two-sample
   Kolmogorov–Smirnov test per feature and returns a per-feature dict of `{statistic, p_value, drifted}`
   (`drifted = p_value < alpha`) — then a report of which features drift on the artificially shifted
   fixture, and confirmation that it reports *no* drift when comparing the reference set against a
   fresh, un-drifted holdout split.
4. Five `test_*` functions (pytest-style: plain `assert` statements, called manually) that exercise all
   of the above against the fixtures, each printing `PASS`/`FAIL`, ending in a short printed QA report
   (how many passed / failed).
5. A 3–5 sentence write-up: what each test caught (or would have caught in production), and why a
   release process that only checks "did the model load and predict" would have missed all three
   failure modes.

Try it yourself in the starter cells below before you scroll down to the solution.

## Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

RANDOM_STATE = 42
pd.set_option("display.width", 120)

## Load the data and train the baseline model (given — this is fixture setup, not the exercise)

In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data          # 30 numeric features, all non-negative measurements
y = data.target        # 0 = malignant, 1 = benign

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

model = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=RANDOM_STATE, n_jobs=-1)
model.fit(X_train, y_train)

print(X.shape)
print("class balance (train):", y_train.value_counts(normalize=True).round(3).to_dict())
X.describe().T[["min", "max", "mean"]]

## Build the test fixtures (given — these are the synthetic, controlled-shift inputs your tests will run against)

- `imbalanced_batch`: a batch of the *test* set artificially resampled to be 95% benign / 5% malignant,
  so overall accuracy is dominated by the majority class.
- `drifted_batch`: a copy of the test features with `mean radius`, `mean texture`, and `mean smoothness`
  shifted by +1.5 standard deviations, simulating a sensor/measurement shift. Everything else is
  untouched.
- `fresh_reference_batch`: a second random split used only to confirm your drift detector reports
  *no* drift between two genuinely un-shifted samples (a sanity check against false positives).

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

# imbalanced_batch: 95% benign / 5% malignant, built from the test set
benign_idx = y_test[y_test == 1].index
malignant_idx = y_test[y_test == 0].index
n_malignant_keep = max(1, int(0.05 / 0.95 * len(benign_idx)))
malignant_sample = rng.choice(malignant_idx, size=min(n_malignant_keep, len(malignant_idx)), replace=False)
imbalanced_idx = np.concatenate([benign_idx.values, malignant_sample])
X_imbalanced, y_imbalanced = X_test.loc[imbalanced_idx], y_test.loc[imbalanced_idx]

# drifted_batch: shift three features by +1.5 std, leave the rest alone
DRIFT_SHIFT_COLS = ["mean radius", "mean texture", "mean smoothness"]
X_drifted = X_test.copy()
for col in DRIFT_SHIFT_COLS:
    X_drifted[col] = X_drifted[col] + 1.5 * X_train[col].std()

# fresh_reference_batch: another random split, no injected shift at all
X_ref_a, X_ref_b, _, _ = train_test_split(X, y, test_size=0.25, random_state=7, stratify=y)

print("imbalanced_batch class balance:", y_imbalanced.value_counts(normalize=True).round(3).to_dict())
print("drifted columns:", DRIFT_SHIFT_COLS)

## Your task

Fill in the TODOs below. Function signatures are given; the logic is up to you.

In [ ]:
class DataValidator:
    """Schema/range/missing-value checks for a DataFrame of model input features."""

    def validate_schema(self, df, expected_columns=tuple(X.columns)):
        # TODO: return a list of violation strings, one per missing or unexpected column.
        # Empty list means the schema matches `expected_columns` exactly (same set, order doesn't matter).
        raise NotImplementedError

    def validate_ranges(self, df, feature_ranges):
        # TODO: feature_ranges is a dict {column: (min_allowed, max_allowed)}.
        # Return a list of violation strings describing any column with values outside its range
        # (include the column name and how many rows violate it).
        raise NotImplementedError

    def validate_missing(self, df):
        # TODO: return a list of violation strings for any column containing NaN/null values
        # (include the column name and the count of missing values).
        raise NotImplementedError


In [ ]:
def evaluate_model(model, X, y):
    """Return a dict with 'accuracy', 'report' (classification_report output_dict),
    and 'confusion_matrix' (as a plain nested list) for `model` on (X, y).
    """
    # TODO:
    # 1. preds = model.predict(X)
    # 2. accuracy = accuracy_score(y, preds)
    # 3. report = classification_report(y, preds, output_dict=True)
    # 4. cm = confusion_matrix(y, preds)
    # 5. return {"accuracy": accuracy, "report": report, "confusion_matrix": cm.tolist()}
    raise NotImplementedError


In [ ]:
def detect_drift(reference_df, current_df, alpha=0.05):
    """Run a two-sample Kolmogorov-Smirnov test per shared column between reference_df and
    current_df. Return a dict {column: {"statistic": float, "p_value": float, "drifted": bool}},
    where drifted = p_value < alpha.
    """
    # TODO:
    # 1. For each column present in both DataFrames, run scipy.stats.ks_2samp(reference_df[col], current_df[col]).
    # 2. Build the per-column result dict described above.
    raise NotImplementedError


In [ ]:
# TODO: write 5 test_* functions, each using plain `assert` statements and printing
# "PASS: <name>" on success or "FAIL: <name> - <reason>" on failure (catch AssertionError, don't
# let it crash the whole run). Suggested tests:
#
# 1. test_schema_valid_input_passes()      -- DataValidator().validate_schema(X_test) is []
# 2. test_schema_catches_missing_column()  -- drop a column, validate_schema flags it
# 3. test_ranges_catch_out_of_bounds()     -- inject an impossible negative value, validate_ranges flags it
# 4. test_accuracy_paradox_on_imbalanced_batch() -- evaluate_model on imbalanced_batch: overall
#    accuracy is high (>0.90) but malignant-class (label 0) recall is low (<0.80) -- assert both
# 5. test_drift_detected_on_shifted_features_only() -- detect_drift(X_train, X_drifted) flags exactly
#    the DRIFT_SHIFT_COLS (or a superset that still includes all of them) as drifted, while
#    detect_drift(X_ref_a, X_ref_b) flags none (or very few, by chance) of them
#
# Then call all 5 and print a summary report: "X/5 tests passed".


### Write-up

TODO: in 3–5 sentences, say what each test caught (or would have caught in production), and why a release process that only checks "did the model load and predict" would have missed all three failure modes.

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
# 1. DataValidator
class DataValidator:
    def validate_schema(self, df, expected_columns=tuple(X.columns)):
        violations = []
        missing = set(expected_columns) - set(df.columns)
        extra = set(df.columns) - set(expected_columns)
        if missing:
            violations.append(f"missing columns: {sorted(missing)}")
        if extra:
            violations.append(f"unexpected columns: {sorted(extra)}")
        return violations

    def validate_ranges(self, df, feature_ranges):
        violations = []
        for col, (lo, hi) in feature_ranges.items():
            if col not in df.columns:
                continue
            bad = (df[col] < lo) | (df[col] > hi)
            if bad.any():
                violations.append(f"{col}: {int(bad.sum())} row(s) outside [{lo}, {hi}]")
        return violations

    def validate_missing(self, df):
        violations = []
        na_counts = df.isna().sum()
        for col, n in na_counts[na_counts > 0].items():
            violations.append(f"{col}: {int(n)} missing value(s)")
        return violations

# 2. evaluate_model
def evaluate_model(model, X, y):
    preds = model.predict(X)
    accuracy = accuracy_score(y, preds)
    report = classification_report(y, preds, output_dict=True)
    cm = confusion_matrix(y, preds)
    return {"accuracy": accuracy, "report": report, "confusion_matrix": cm.tolist()}

# 3. detect_drift
def detect_drift(reference_df, current_df, alpha=0.05):
    result = {}
    shared_cols = [c for c in reference_df.columns if c in current_df.columns]
    for col in shared_cols:
        stat, p_value = ks_2samp(reference_df[col], current_df[col])
        result[col] = {"statistic": float(stat), "p_value": float(p_value), "drifted": p_value < alpha}
    return result

# 4. feature_ranges derived from the training data (what "valid" looks like)
FEATURE_RANGES = {col: (0.0, float(X_train[col].max()) * 1.2) for col in X_train.columns}

# 5. test_* functions
def test_schema_valid_input_passes():
    name = "test_schema_valid_input_passes"
    try:
        violations = DataValidator().validate_schema(X_test)
        assert violations == [], f"expected no violations, got {violations}"
        print(f"PASS: {name}")
        return True
    except AssertionError as e:
        print(f"FAIL: {name} - {e}")
        return False

def test_schema_catches_missing_column():
    name = "test_schema_catches_missing_column"
    try:
        broken = X_test.drop(columns=["mean radius"])
        violations = DataValidator().validate_schema(broken)
        assert any("mean radius" in v for v in violations), f"expected 'mean radius' flagged, got {violations}"
        print(f"PASS: {name}")
        return True
    except AssertionError as e:
        print(f"FAIL: {name} - {e}")
        return False

def test_ranges_catch_out_of_bounds():
    name = "test_ranges_catch_out_of_bounds"
    try:
        broken = X_test.copy()
        broken.iloc[0, broken.columns.get_loc("mean radius")] = -999.0
        violations = DataValidator().validate_ranges(broken, FEATURE_RANGES)
        assert any("mean radius" in v for v in violations), f"expected 'mean radius' flagged, got {violations}"
        print(f"PASS: {name}")
        return True
    except AssertionError as e:
        print(f"FAIL: {name} - {e}")
        return False

def test_accuracy_paradox_on_imbalanced_batch():
    name = "test_accuracy_paradox_on_imbalanced_batch"
    try:
        result = evaluate_model(model, X_imbalanced, y_imbalanced)
        malignant_recall = result["report"]["0"]["recall"]
        assert result["accuracy"] > 0.90, f"expected high overall accuracy, got {result['accuracy']:.3f}"
        assert malignant_recall < 0.80, f"expected degraded malignant recall, got {malignant_recall:.3f}"
        print(f"PASS: {name} (accuracy={result['accuracy']:.3f}, malignant recall={malignant_recall:.3f})")
        return True
    except AssertionError as e:
        print(f"FAIL: {name} - {e}")
        return False

def test_drift_detected_on_shifted_features_only():
    name = "test_drift_detected_on_shifted_features_only"
    try:
        drift_result = detect_drift(X_train, X_drifted)
        flagged = {c for c, r in drift_result.items() if r["drifted"]}
        assert set(DRIFT_SHIFT_COLS).issubset(flagged), f"expected {DRIFT_SHIFT_COLS} flagged, got {sorted(flagged)}"

        sanity_result = detect_drift(X_ref_a, X_ref_b)
        sanity_flagged = {c for c, r in sanity_result.items() if r["drifted"]}
        assert len(sanity_flagged) <= 2, f"expected ~no false drift on unshifted split, got {sorted(sanity_flagged)}"
        print(f"PASS: {name} (flagged={sorted(flagged)}, false positives={sorted(sanity_flagged)})")
        return True
    except AssertionError as e:
        print(f"FAIL: {name} - {e}")
        return False

tests = [
    test_schema_valid_input_passes,
    test_schema_catches_missing_column,
    test_ranges_catch_out_of_bounds,
    test_accuracy_paradox_on_imbalanced_batch,
    test_drift_detected_on_shifted_features_only,
]
results = [t() for t in tests]
print(f"\n{sum(results)}/{len(results)} tests passed")
```

**Typical output**

```
PASS: test_schema_valid_input_passes
PASS: test_schema_catches_missing_column
PASS: test_ranges_catch_out_of_bounds
PASS: test_accuracy_paradox_on_imbalanced_batch (accuracy=0.961, malignant recall=0.500)
PASS: test_drift_detected_on_shifted_features_only (flagged=['mean radius', 'mean smoothness', 'mean texture'], false positives=[])

5/5 tests passed
```

**Explanation**

The schema and range tests catch exactly the kind of bad input a real serving pipeline sees after a
field gets renamed, dropped, or corrupted upstream — cheap checks that run in milliseconds and would
have stopped a broken request before it ever reached `model.predict`. The accuracy-paradox test shows
why a release gate built only around overall accuracy is dangerous for an imbalanced problem: on the
95/5 batch the model reports ~96% accuracy — a number any dashboard would wave through — while its
recall on the malignant class (the class that actually matters for a cancer screen) drops to around
50%, i.e. it is barely better than a coin flip at catching the cases you most need it to catch. The
drift test demonstrates that a purely statistical check, with no retraining and no labels required, can
flag exactly the three features that were shifted while staying quiet on a second, genuinely unshifted
split — which is the difference between a drift detector that's useful in production and one that cries
wolf constantly. A release process that only checks "did the model load and predict without throwing"
would pass all three of these broken scenarios: the schema violations would either crash downstream or
silently coerce bad values, the imbalanced-batch model would "work" (return predictions, report high
accuracy) while being clinically useless, and the drifted-batch model would keep serving confident,
silently degrading predictions with no alarm at all.

</details>